# Lab 2 — India Air Quality & Crop Yield: Exploratory Data Analysis

**Course:** Machine Learning Lab  
**Institution:** Christ University  
**Datasets:**
- `city_day.csv` — Daily AQI readings for 26 Indian cities, January 2015 – July 2020 (29,531 rows)
- `crop_production.csv` — District-level crop production across 33 Indian states, 1997–2015 (246,091 rows)

**Tasks:** 6 through 9 (Tasks A–C optional/bonus)  
**Marks:** 10 (core) + bonus

---

> **Note on temporal overlap:** The AQI dataset spans 2015–2020; the crop dataset spans 1997–2015. The only nominal overlap is 2015, and within that year the crop data has entries for only two states (Odisha and Sikkim), neither of which has AQI stations in this dataset. For Tasks 6–7, the full AQI time series is used. For Task 8, datasets are merged at the state level without a year constraint — each state's multi-year mean AQI (2015–2020) is paired with its cumulative crop metrics (1997–2015). This mismatch is flagged where relevant.

## Setup: Imports and Preprocessing

Steps:
1. Load both CSVs using relative paths.
2. Parse `Date`; extract `Year` and `Month`.
3. Map each city to its state via the city→state dictionary.
4. Drop rows with missing AQI.
5. Impute remaining numeric pollutant columns with column medians.
6. Standardise state names in both datasets with `str.strip().str.title()`.

In [ ]:
# ── Standard library & third-party imports ──────────────────────────────────
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats

warnings.filterwarnings('ignore')

# ── Plot style ───────────────────────────────────────────────────────────────
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams.update({'figure.dpi': 120, 'axes.titlesize': 13,
                     'axes.labelsize': 11, 'figure.autolayout': True})

# ── 1. Load raw data (relative paths — run notebook from its own directory) ─
aqi_raw  = pd.read_csv('city_day.csv')
crop_raw = pd.read_csv('crop_production.csv')

print(f'AQI raw shape  : {aqi_raw.shape}')
print(f'Crop raw shape : {crop_raw.shape}')

In [ ]:
# ── 2. City → State mapping (Lab 1 preprocessing) ───────────────────────────
CITY_TO_STATE = {
    'Ahmedabad'       : 'Gujarat',
    'Aizawl'          : 'Mizoram',
    'Amaravati'       : 'Andhra Pradesh',
    'Amritsar'        : 'Punjab',
    'Bengaluru'       : 'Karnataka',
    'Bhopal'          : 'Madhya Pradesh',
    'Brajrajnagar'    : 'Odisha',
    'Chandigarh'      : 'Chandigarh',
    'Chennai'         : 'Tamil Nadu',
    'Coimbatore'      : 'Tamil Nadu',
    'Delhi'           : 'Delhi',
    'Ernakulam'       : 'Kerala',
    'Gurugram'        : 'Haryana',
    'Guwahati'        : 'Assam',
    'Hyderabad'       : 'Telangana',
    'Jaipur'          : 'Rajasthan',
    'Jorapokhar'      : 'Jharkhand',
    'Kochi'           : 'Kerala',
    'Kolkata'         : 'West Bengal',
    'Lucknow'         : 'Uttar Pradesh',
    'Mumbai'          : 'Maharashtra',
    'Patna'           : 'Bihar',
    'Shillong'        : 'Meghalaya',
    'Talcher'         : 'Odisha',
    'Thiruvananthapuram': 'Kerala',
    'Visakhapatnam'   : 'Andhra Pradesh',
}

# ── 3. AQI preprocessing ────────────────────────────────────────────────────
aqi = aqi_raw.copy()

# Parse date; extract temporal features
aqi['Date']  = pd.to_datetime(aqi['Date'])
aqi['Year']  = aqi['Date'].dt.year
aqi['Month'] = aqi['Date'].dt.month

# Map city to state
aqi['State'] = aqi['City'].map(CITY_TO_STATE)

# Drop rows with missing AQI (target variable — cannot impute meaningfully)
aqi.dropna(subset=['AQI'], inplace=True)

# Impute remaining numeric pollutant columns with column medians
pollutant_cols = ['PM2.5', 'PM10', 'NO', 'NO2', 'NOx', 'NH3',
                  'CO', 'SO2', 'O3', 'Benzene', 'Toluene', 'Xylene']
for col in pollutant_cols:
    aqi[col].fillna(aqi[col].median(), inplace=True)

# Standardise state names
aqi['State'] = aqi['State'].str.strip().str.title()

print(f'AQI cleaned shape : {aqi.shape}')
print(f'Year range        : {aqi["Year"].min()} – {aqi["Year"].max()}')
print(f'Cities            : {aqi["City"].nunique()}')
print(f'States            : {aqi["State"].nunique()}')
aqi.head(3)

In [ ]:
# ── 4. Crop preprocessing ────────────────────────────────────────────────────
crop = crop_raw.copy()

# Drop rows with missing Production (no sensible imputation for yield)
crop.dropna(subset=['Production'], inplace=True)

# Standardise state names to match AQI state names
crop['State_Name'] = crop['State_Name'].str.strip().str.title()

print(f'Crop cleaned shape : {crop.shape}')
print(f'Year range         : {crop["Crop_Year"].min()} – {crop["Crop_Year"].max()}')
print(f'States             : {crop["State_Name"].nunique()}')
print(f'Unique crops       : {crop["Crop"].nunique()}')
crop.head(3)

---
## Task 6 — AQI Trend Over Time (2 marks)

**Question:** Has India's air quality improved or worsened between 2015 and 2020?

**Approach:** AQI data is grouped by `Year` and mean AQI is computed across all 26 cities. The result is plotted as a line chart with the best and worst years annotated. Lower AQI = cleaner air.

> **Note on 2020:** Data runs only to July 1, so the 2020 mean covers January–June only. This is marked on the chart.

In [ ]:
# ── Task 6: AQI Trend Over Time ──────────────────────────────────────────────

# Group by Year and compute mean AQI
yearly_aqi = aqi.groupby('Year')['AQI'].mean().reset_index()
yearly_aqi.columns = ['Year', 'Mean_AQI']

# Identify years with highest and lowest mean AQI
max_year = yearly_aqi.loc[yearly_aqi['Mean_AQI'].idxmax()]
min_year = yearly_aqi.loc[yearly_aqi['Mean_AQI'].idxmin()]

print('Yearly mean AQI:')
print(yearly_aqi.to_string(index=False))
print(f'\nHighest AQI: {max_year["Mean_AQI"]:.1f} in {int(max_year["Year"])}')
print(f'Lowest AQI : {min_year["Mean_AQI"]:.1f} in {int(min_year["Year"])}')

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

# Main trend line
ax.plot(yearly_aqi['Year'], yearly_aqi['Mean_AQI'],
        color='steelblue', linewidth=2.5, marker='o',
        markersize=7, label='Annual mean AQI')

# Highlight worst year (red)
ax.scatter(max_year['Year'], max_year['Mean_AQI'],
           color='crimson', s=130, zorder=5, label=f'Worst year ({int(max_year["Year"])}: AQI={max_year["Mean_AQI"]:.1f})')
ax.annotate(f' {int(max_year["Year"])}\n AQI {max_year["Mean_AQI"]:.1f}',
            xy=(max_year['Year'], max_year['Mean_AQI']),
            xytext=(max_year['Year'] + 0.1, max_year['Mean_AQI'] + 5),
            color='crimson', fontsize=9.5, fontweight='bold')

# Highlight best year (green)
ax.scatter(min_year['Year'], min_year['Mean_AQI'],
           color='seagreen', s=130, zorder=5, label=f'Best year ({int(min_year["Year"])}: AQI={min_year["Mean_AQI"]:.1f})')
ax.annotate(f' {int(min_year["Year"])}\n AQI {min_year["Mean_AQI"]:.1f}',
            xy=(min_year['Year'], min_year['Mean_AQI']),
            xytext=(min_year['Year'] - 0.6, min_year['Mean_AQI'] - 15),
            color='seagreen', fontsize=9.5, fontweight='bold')

# Note partial year 2020
ax.axvspan(2019.5, 2020.5, alpha=0.08, color='orange', label='2020: Jan–Jun only')

ax.set_title('India Mean Annual AQI — 2015 to 2020\n(26 monitoring cities across 21 states)', pad=12)
ax.set_xlabel('Year')
ax.set_ylabel('Mean AQI')
ax.set_xticks(yearly_aqi['Year'])
ax.legend(framealpha=0.85, fontsize=9)
ax.grid(axis='y', alpha=0.4)

plt.tight_layout()
plt.savefig('task6_aqi_trend.png', dpi=120, bbox_inches='tight')
plt.show()
print('Chart saved to task6_aqi_trend.png')

### Observations

Mean AQI dropped from ~212 in 2015 to ~114 in the first half of 2020 — a fall of nearly 100 points over five years. 2015 sits in the "Poor" category on India's national index; 2020 is closer to "Moderate".

The sharpest single-year drop was between 2018 and 2019 (183 → 157).

The 2020 figure covers only January–June, which are generally cleaner months — a full-year average would likely be higher.

---
## Task 7 — Seasonal AQI Pattern (2 marks)

**Question:** An NGO claims AQI is worst during October–December due to post-harvest crop residue burning. Does the data support this?

**Approach:** AQI readings are grouped by `Month` and mean AQI is computed for each month, pooling all years and cities. A bar chart shows the seasonal pattern with a reference line at the overall dataset mean. Months above the mean are coloured differently for quick comparison.

In [ ]:
# ── Task 7: Seasonal AQI Pattern ─────────────────────────────────────────────

import calendar

# Group by month, compute mean AQI
monthly_aqi = aqi.groupby('Month')['AQI'].mean().reset_index()
monthly_aqi.columns = ['Month', 'Mean_AQI']

# Convert month number to abbreviated name for the chart
monthly_aqi['Month_Name'] = monthly_aqi['Month'].apply(
    lambda m: calendar.month_abbr[m]
)

# Overall mean AQI reference line
overall_mean = aqi['AQI'].mean()

print('Monthly mean AQI (all years combined):')
print(monthly_aqi[['Month_Name', 'Mean_AQI']].to_string(index=False))
print(f'\nOverall dataset mean AQI: {overall_mean:.1f}')

# Identify the worst 3-month window
worst3 = monthly_aqi.nlargest(3, 'Mean_AQI')
print('\nThree highest-pollution months:')
print(worst3[['Month_Name', 'Mean_AQI']].to_string(index=False))

In [ ]:
# Colour bars: above-average = darker orange, below-average = muted blue
bar_colours = [
    '#d95f02' if v > overall_mean else '#7fbfdf'
    for v in monthly_aqi['Mean_AQI']
]

fig, ax = plt.subplots(figsize=(11, 5))

bars = ax.bar(monthly_aqi['Month_Name'], monthly_aqi['Mean_AQI'],
              color=bar_colours, edgecolor='white', linewidth=0.6, width=0.7)

# Value labels on top of each bar
for bar in bars:
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 3,
            f'{bar.get_height():.0f}',
            ha='center', va='bottom', fontsize=8.5, color='#333333')

# Overall mean reference line
ax.axhline(overall_mean, color='#333333', linestyle='--', linewidth=1.4,
           label=f'Overall mean AQI ({overall_mean:.0f})')

# Shade the Oct–Dec window the NGO referred to
ax.axvspan(8.5, 11.5, alpha=0.10, color='red',
           label='NGO-flagged window (Oct–Dec)')

# Custom legend patch for bar colours
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#d95f02', label='Above overall mean'),
    Patch(facecolor='#7fbfdf', label='Below overall mean'),
    plt.Line2D([0], [0], color='#333333', linestyle='--', linewidth=1.4,
               label=f'Overall mean ({overall_mean:.0f})'),
    Patch(facecolor='red', alpha=0.25, label='NGO window (Oct–Dec)'),
]
ax.legend(handles=legend_elements, fontsize=9, framealpha=0.85)

ax.set_title('Average AQI by Month — India (2015–2020)\n'
             'Orange bars = above overall mean; data pooled across all 26 cities', pad=12)
ax.set_xlabel('Month')
ax.set_ylabel('Mean AQI')
ax.set_ylim(0, monthly_aqi['Mean_AQI'].max() * 1.18)
ax.grid(axis='y', alpha=0.35)

plt.tight_layout()
plt.savefig('task7_seasonal_aqi.png', dpi=120, bbox_inches='tight')
plt.show()
print('Chart saved to task7_seasonal_aqi.png')

### Observations

The NGO's claim is broadly correct, but the peak is slightly earlier than October–December.

- **November** is the single worst month (mean AQI ≈ 242), followed by January (≈ 232) and December (≈ 227).
- October (≈ 189) is above the overall mean of 166 — air quality does deteriorate from October onward.
- The cleanest months are **July and August** (AQI ≈ 112–114), coinciding with the monsoon season when rainfall clears particulates.
- Air quality worsens from September as monsoon rains withdraw, paddy harvests end, and stubble burning begins across northern states.

**Conclusion:** The Oct–Dec window is genuinely above average, supporting the NGO's claim. However, November is the true peak, and January is nearly as bad — winter temperature inversions are likely an equally important driver alongside crop burning.

---
## Task 8 — Merge Datasets & Correlation Analysis (3 marks)

**Question:** Is there a measurable relationship between a state's air pollution level and its agricultural output?

### Why aggregation is needed

The two datasets are at different granularities:

| Dataset | Grain | Geography | Time |
|---------|-------|-----------|------|
| `city_day.csv` | One row per city per day | 26 cities | 2015–2020 |
| `crop_production.csv` | One row per district per season per crop | 33 states | 1997–2015 |

To join them:
1. Aggregate AQI from city-day to **state-level** using the city→state map.
2. Aggregate crop data from district-season-crop to **state-level** by summing `Area` and `Production`.
3. Join on `State` (after standardising naming in both datasets).

### Temporal limitation

> The AQI dataset covers **2015–2020**; the crop dataset covers **1997–2015**. A year-level inner join returns zero rows — within the nominal overlap year (2015), only Odisha and Sikkim have crop entries, and neither has AQI stations in this dataset.
>
> The merge is done at the **state level only**, pairing each state's mean AQI (2015–2020) with its cumulative crop metrics (1997–2015). This yields a 20-state dataset, but any relationship found is descriptive only — not causal or contemporaneous.

In [ ]:
# ── Task 8, Step 1–3: Build state-level summaries for both datasets ───────────

# State-level mean AQI (across all years 2015–2020)
state_aqi = (
    aqi.groupby('State')['AQI']
       .mean()
       .reset_index()
       .rename(columns={'AQI': 'Mean_AQI'})
)

# State-level crop totals (cumulative 1997–2015)
state_crop = (
    crop.groupby('State_Name')
        .agg(Total_Area=('Area', 'sum'),
             Total_Production=('Production', 'sum'))
        .reset_index()
        .rename(columns={'State_Name': 'State'})
)

print(f'AQI state summary  : {state_aqi.shape[0]} states')
print(f'Crop state summary : {state_crop.shape[0]} states')

In [ ]:
# ── Task 8, Step 4: Standardise state names ───────────────────────────────────
# Both datasets have already had .str.strip().str.title() applied in preprocessing.
# Apply a rename map for any historical variants that differ between datasets.

STATE_RENAME = {
    # crop dataset name  →  AQI-consistent name
    'Orissa'       : 'Odisha',      # older spelling
    'Uttaranchal'  : 'Uttarakhand', # renamed in 2007
}

state_aqi['State']  = state_aqi['State'].replace(STATE_RENAME)
state_crop['State'] = state_crop['State'].replace(STATE_RENAME)

print('AQI states  :', sorted(state_aqi['State'].tolist()))
print()
print('Crop states :', sorted(state_crop['State'].tolist()))

In [ ]:
# ── Task 8, Step 5–6: Merge and inspect ──────────────────────────────────────

merged = pd.merge(state_aqi, state_crop, on='State', how='inner')

print(f'Merged DataFrame shape: {merged.shape}')
print(f'States included       : {sorted(merged["State"].tolist())}')
print()
print(merged.head(10).to_string(index=False))

In [ ]:
# ── Task 8, Step 7: Correlation heatmap ──────────────────────────────────────

numeric_cols = ['Mean_AQI', 'Total_Area', 'Total_Production']
corr_matrix  = merged[numeric_cols].corr()

print('Correlation matrix:')
print(corr_matrix.round(4).to_string())

fig, ax = plt.subplots(figsize=(6, 5))

sns.heatmap(
    corr_matrix,
    annot=True,
    fmt='.3f',
    cmap='RdYlGn_r',   # red = positive, green = negative
    vmin=-1, vmax=1,
    square=True,
    linewidths=0.8,
    linecolor='white',
    annot_kws={'size': 13, 'weight': 'bold'},
    ax=ax
)

ax.set_title('Pearson Correlation — State-Level AQI vs Crop Metrics\n'
             '(20 states; AQI: 2015–2020 mean; Crop: 1997–2015 cumulative)', pad=12)
ax.set_xticklabels(['Mean AQI', 'Total Area\n(ha)', 'Total\nProduction (T)'],
                   rotation=0)
ax.set_yticklabels(['Mean AQI', 'Total Area (ha)', 'Total Production (T)'],
                   rotation=0)

plt.tight_layout()
plt.savefig('task8_correlation_heatmap.png', dpi=120, bbox_inches='tight')
plt.show()
print('Chart saved to task8_correlation_heatmap.png')

### Observations

**AQI vs Total Production: r ≈ −0.19 (weak negative)**  
States with higher mean AQI tend to have slightly lower total crop production. Southern states like Kerala, Tamil Nadu, and Karnataka have relatively clean air but produce large volumes of coconut, banana, and spices (high by weight), which pulls the relationship negative. The correlation is weak and likely driven by crop type and state size rather than any direct effect of pollution on yield.

**AQI vs Total Area: r ≈ +0.24 (weak positive)**  
States that cultivate more land tend to have marginally higher AQI. Large agricultural states in the Indo-Gangetic Plain (Uttar Pradesh, Madhya Pradesh, Rajasthan) also host heavy industry, dense traffic, and biomass burning — all coincidental pollution drivers bundled with geographic scale.

Both correlations are small and likely reflect confounding variables rather than a direct link between pollution and agricultural output.

---
## Task 9 — Minister's Briefing (3 marks)

---

**BRIEFING NOTE**

**Date:** June 2026  
**To:** The Honourable State Environment Minister  
**From:** Data Analysis Unit, Environment Research Division  
**Subject:** Key findings — India Air Quality and Crop Production study (AQI: 2015–2020; Crop: 1997–2015)

---

**Honourable Minister,**

Three findings from the analysis of air quality readings across 26 Indian cities and crop production records across 33 states are summarised below.

**Finding 1 — Air quality is improving.** Mean AQI across monitored cities fell from 212 in 2015 to 114 in the first half of 2020 — a drop of nearly 100 points in five years.

**Finding 2 — Winter months are the most polluted.** November (mean AQI 242) and January (232) are the worst months, both well above the annual average of 166. October through December are all above average, consistent with crop residue burning and winter air trapping pollutants near the ground.

**Finding 3 — The most polluted states are Gujarat, Bihar, Haryana, Delhi, and Uttar Pradesh.** States in the Indo-Gangetic Plain carry the highest pollution burden. Mizoram, Meghalaya, and Kerala record the cleanest air.

**Recommendation:** Introduce a targeted subsidy in Punjab, Haryana, and Uttar Pradesh for mechanical crop-residue mulchers. Reducing stubble burning in October and November would directly address the seasonal peak identified in Finding 2.

**Limitation:** High-pollution states show slightly lower crop output (r = −0.19), but this correlation is weak and shaped by confounders — rainfall, irrigation, soil quality, and crop type are not captured in this dataset. This correlation alone should not drive policy decisions.

*Respectfully submitted,*  
*Data Analysis Unit*

---
## Lab Summary

| Task | Topic | Key Finding |
|------|-------|-------------|
| 6    | AQI trend over time | Mean AQI fell from 212 (2015) to 114 (Jan–Jun 2020); consistent year-on-year improvement |
| 7    | Seasonal AQI | November is worst (AQI ≈ 242); Oct–Dec all above average; July–August cleanest |
| 8    | Cross-dataset merge & correlation | 20 states matched; AQI vs Production r ≈ −0.19 (weak, not significant); temporal gap limits inference |
| 9    | Minister's briefing | Stubble-burning subsidy recommended; correlation ≠ causation explicitly stated |

**Key limitation:** The most important constraint is the temporal mismatch between datasets. A proper pollution–yield analysis would require AQI and crop data from the same years at comparable spatial scales.